In [1]:
from xgboost import XGBClassifier
import warnings
import os
import sys
import yaml

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    f1_score,
    recall_score,
    precision_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

warnings.filterwarnings("ignore")
plt.style.use("ggplot")
pd.set_option("display.max_columns", None)

In [2]:
sys.path.append(os.path.abspath('..'))
from scripts.evaluate import evaluate_classifier

CONFIG_PATH = '../configs/paths.yaml'

with open(CONFIG_PATH, 'r') as file:
    paths = yaml.safe_load(file)

df_train = pd.read_csv(paths['features_data']['train_data'])
df_test  = pd.read_csv(paths['features_data']['test_data'])

print(f"Train shape: {df_train.shape}")
print(f"Test shape : {df_test.shape}")

Train shape: (79219, 38)
Test shape : (20124, 38)


In [3]:
X_train = df_train.drop(columns=['readmitted_binary'])
X_test  = df_test.drop(columns=['readmitted_binary'])

y_train = df_train['readmitted_binary']
y_test  = df_test['readmitted_binary']

print(f"X_train: {X_train.shape}  |  y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}   |  y_test : {y_test.shape}")

X_train: (79219, 37)  |  y_train: (79219,)
X_test : (20124, 37)   |  y_test : (20124,)


In [4]:
from scripts.preprocessor import get_preprocessor

preprocessor = get_preprocessor()

In [5]:
y_train.value_counts()

readmitted_binary
0    70246
1     8973
Name: count, dtype: int64

In [6]:
scale_pos_weight = 70246 / 8973

In [ ]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('xgboost',XGBClassifier(       
        random_state=32,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss"
    ))
])

default_result = evaluate_classifier(
    'Gradient Boost (default)',
    pipeline,
    X_train, X_test,
    y_train, y_test
)

print("GradientBoost — Default Parameters")
print(classification_report(y_test, default_result['_test_pred']))

GradientBoost — Default Parameters
              precision    recall  f1-score   support

           0       0.91      0.70      0.79     17783
           1       0.18      0.48      0.26      2341

    accuracy                           0.68     20124
   macro avg       0.54      0.59      0.53     20124
weighted avg       0.83      0.68      0.73     20124



In [8]:
param_grid = {
    "xgboost__n_estimators": [100, 200],
    "xgboost__learning_rate": [0.01, 0.1],
    "xgboost__max_depth": [3, 5, 7],
    "xgboost__subsample": [0.8, 1.0],
    "xgboost__colsample_bytree": [0.8, 1.0],
}

In [9]:
grid = GridSearchCV(
    estimator= pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    verbose=3
)

print("Running coarse grid search...")
grid.fit(X_train, y_train)


Running coarse grid search...
Fitting 5 folds for each of 48 candidates, totalling 240 fits
[CV 1/5] END xgboost__colsample_bytree=0.8, xgboost__learning_rate=0.01, xgboost__max_depth=3, xgboost__n_estimators=100, xgboost__subsample=0.8;, score=0.269 total time=   0.8s
[CV 2/5] END xgboost__colsample_bytree=0.8, xgboost__learning_rate=0.01, xgboost__max_depth=3, xgboost__n_estimators=100, xgboost__subsample=0.8;, score=0.270 total time=   0.9s
[CV 3/5] END xgboost__colsample_bytree=0.8, xgboost__learning_rate=0.01, xgboost__max_depth=3, xgboost__n_estimators=100, xgboost__subsample=0.8;, score=0.267 total time=   1.0s
[CV 4/5] END xgboost__colsample_bytree=0.8, xgboost__learning_rate=0.01, xgboost__max_depth=3, xgboost__n_estimators=100, xgboost__subsample=0.8;, score=0.259 total time=   1.0s
[CV 5/5] END xgboost__colsample_bytree=0.8, xgboost__learning_rate=0.01, xgboost__max_depth=3, xgboost__n_estimators=100, xgboost__subsample=0.8;, score=0.251 total time=   0.9s
[CV 1/5] END xgboo